# Homework Starter — Stage 13: Productization

**This homework is self-contained.** It does not use your project data or your project
model — the cells below generate everything they need. Work through it in order.

You are building four things: a saved model, a Flask app that loads it at startup and
serves two routes, proof from this notebook that both routes work, and a README that
tells someone else how to call them.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install scikit-learn
# !pip install joblib
# !pip install flask
# !pip install requests

## 1. Generate data and train a model

Nothing to fill in here — run it. Note `os.makedirs` **before** `joblib.dump`: without it
the save fails with `FileNotFoundError`, because `model/` does not exist yet.

In [2]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# the dataset for this homework - generated, not loaded
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

model = LinearRegression()
model.fit(X, y)

os.makedirs('model', exist_ok=True)          # BEFORE the dump, not after
joblib.dump(model, 'model/model.pkl')

# prove the file on disk is usable: load it back and predict with the loaded copy
reloaded = joblib.load('model/model.pkl')
print('saved to model/model.pkl')
print('prediction from the reloaded model:', reloaded.predict([[0.1, 0.2]])[0])

saved to model/model.pkl
prediction from the reloaded model: 23.58961171297328


## 2. Write `app.py`

The completed cell writes a Flask application with POST and GET prediction routes, input validation, and one model loaded at startup.


In [3]:
app_code = r'''from flask import Flask, request, jsonify
import joblib

# Loaded once when the application starts.
model = joblib.load('model/model.pkl')
app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}
    features = data.get('features')
    if not isinstance(features, list) or len(features) != 2:
        return jsonify({'error': 'features must be a list of exactly 2 numbers'}), 400
    try:
        values = [float(value) for value in features]
    except (TypeError, ValueError):
        return jsonify({'error': 'features must contain only numbers'}), 400
    prediction = float(model.predict([values])[0])
    return jsonify({'prediction': prediction})


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):
    try:
        values = [float(f1), float(f2)]
    except ValueError:
        return jsonify({'error': 'f1 and f2 must be numbers'}), 400
    prediction = float(model.predict([values])[0])
    return jsonify({'prediction': prediction})


if __name__ == '__main__':
    app.run(port=5055)
'''

with open('app.py', 'w', encoding='utf-8') as file:
    file.write(app_code)
print('wrote app.py')


wrote app.py


## 3. Launch the server

This opens a **separate terminal window** and starts Flask there. Leave it running.
Every time you change `app.py`, close that window and run this cell again.

In [4]:
import subprocess
import sys
import time

server_process = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(2)
print('Flask launched on http://127.0.0.1:5055')


Flask launched on http://127.0.0.1:5055


## 4. Call your own API

Three calls: the POST route, the GET route, and one deliberately bad call. **Leave this
output visible in the notebook you submit — it is your testing evidence.**

In [5]:
import requests

BASE = 'http://127.0.0.1:5055'

try:
    r1 = requests.post(
        BASE + '/predict',
        json={'features': [0.1, 0.2]},
        timeout=5,
    )
    print('POST /predict          ', r1.status_code, r1.text.strip())

    r2 = requests.get(BASE + '/predict/0.1/0.2', timeout=5)
    print('GET  /predict/0.1/0.2  ', r2.status_code, r2.text.strip())

    r3 = requests.get(BASE + '/predict/abc/0.2', timeout=5)
    print('GET  /predict/abc/0.2  ', r3.status_code, r3.text.strip())

    assert r1.status_code == 200
    assert r2.status_code == 200
    assert r3.status_code == 400
    assert 'prediction' in r1.json()
    assert 'prediction' in r2.json()
    assert 'error' in r3.json()
finally:
    if 'server_process' in globals() and server_process.poll() is None:
        server_process.terminate()
        server_process.wait(timeout=5)
        print('Flask server stopped after tests.')


POST /predict           200 {"prediction":23.58961171297328}
GET  /predict/0.1/0.2   200 {"prediction":23.58961171297328}
GET  /predict/abc/0.2   400 {"error":"f1 and f2 must be numbers"}
Flask server stopped after tests.


## 5. Write `README.md`

The completed cell writes reproducible startup instructions, copy-pasteable examples for both routes, real responses from the tests above, and bad-input behavior.


In [6]:
post_response = r1.text.strip()
get_response = r2.text.strip()
bad_response = r3.text.strip()

readme = f'''# Stage 13 Homework - Prediction API

This Flask API serves predictions from a two-feature linear regression model trained on a deterministic synthetic dataset. The model is loaded once when the application starts and is reused by both prediction routes.

## Install dependencies

    python -m pip install scikit-learn joblib flask requests

## Running it

    python app.py

The server starts on http://127.0.0.1:5055 and loads `model/model.pkl` at startup.

## POST /predict

    curl -X POST http://127.0.0.1:5055/predict \
         -H "Content-Type: application/json" \
         -d '{{"features": [0.1, 0.2]}}'

Response: `{post_response}`

## GET /predict/<f1>/<f2>

    curl http://127.0.0.1:5055/predict/0.1/0.2

Response: `{get_response}`

## Bad input

Missing features, the wrong number of features, nonnumeric JSON values, or nonnumeric path parameters return HTTP 400 with a JSON error. Example: `{bad_response}`
'''

with open('README.md', 'w', encoding='utf-8') as file:
    file.write(readme)
print('wrote completed README.md')


wrote completed README.md


### Save Notebook
Remember to save as `homework13_productization_submission.ipynb`.